# Fase 15 — Hipótese: remover `comps_knn_price` melhora o modelo?

**Motivação:** `comps_knn_price` domina 40,5% do gain total (gain-based, XGBoost) — mais que o dobro da
2ª feature (`sqft_living`, 19,1%) e quase tanto quanto as outras 19 features somadas (~59,5%). Essa
discrepância motiva a pergunta: será que o modelo generaliza melhor sem ela (menos dependência de uma
única feature target-derived, fase 06)?

**Critério pré-registrado (P4 — definido ANTES de rodar, nunca ajustado depois do resultado):**

- **TRAIN, `GroupKFold(3)`** (mesmo método fases 09/10, hiperparâmetros oficiais do modelo final —
  XGBoost `n_estimators=400, max_depth=3, learning_rate=0.05`): MAE do candidato (19 features) **não
  pode piorar** em relação ao baseline (20 features).
- **TEST** (modelo fit só em TRAIN, aplicado em TEST nunca visto — mesmo método fase 11/12): métrica
  **decisiva** para generalização (mesmo precedente da fase 12 iteração 2, que usou TEST
  deliberadamente quando a pergunta é sobre generalização, não sobre CV em TRAIN). MAE precisa **cair
  ≥5%** para adotar.
- **Adoção exige as duas condições** — TRAIN não piora E TEST melhora ≥5%. Resultado misto (ex: TEST
  melhora mas TRAIN piora) é reportado como inconclusivo, não adotado sem uma nova rodada de análise.
- **Quebra por `price_band` (P3)** — nunca decide só pelo MAE global; uma melhora agregada que esconde
  piora em `Luxury` é motivo de rejeição, mesmo que o número global passe no critério (mesma disciplina
  do critério de substituição de `docs/08_continuous_learning.md`, seção 4). Nenhuma banda pode piorar
  mais que 10%.

**Não é uma fase que decide sozinha** — se ADOTADA, a hipótese só vira modelo oficial depois de refazer
fase 10 (seleção) + fase 13 (refit final + checagem em `val`) + fase 14 (contrato) com o conjunto de 19
features, feito por `scripts/promote_hypothesis_drop_comps_knn_price.py`.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / ".git").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)

In [ ]:
from src.scripts.run_hypothesis_drop_comps_knn_price import run_drop_comps_knn_price_hypothesis_pipeline

# A própria função já persiste `reports/hypothesis_drop_comps_knn_price.json` (mesmo comportamento do
# `main()` do script) — nada para gravar manualmente aqui.
report = run_drop_comps_knn_price_hypothesis_pipeline()

print("decision:", report["decision"])
print("rejection_reasons:", report["rejection_reasons"])

decision: REJECT_KEEP_COMPS_KNN_PRICE
rejection_reasons: ['TRAIN GroupKFold(3) MAE piorou +34.48% (critério: não pode piorar)', 'TEST MAE mudou -4.07%, não atingiu a queda decisiva de -5% (near miss — quase bateu o critério, mesmo padrão da fase 12 iter.1)', "banda(s) pioraram >10%: {'Entry': 0.3461739050600445}"]


In [3]:
import pandas as pd

comparison_df = pd.DataFrame(
    [
        {
            "feature_set": "baseline_20_features",
            "n_features": report["feature_counts"]["baseline"],
            "train_groupkfold3_mae": report["train_groupkfold3"]["baseline_mae_mean"],
            "test_mae": report["test_holdout"]["baseline_mae"],
        },
        {
            "feature_set": "sem_comps_knn_price_19_features",
            "n_features": report["feature_counts"]["candidato"],
            "train_groupkfold3_mae": report["train_groupkfold3"]["candidato_mae_mean"],
            "test_mae": report["test_holdout"]["candidato_mae"],
        },
    ]
)
comparison_df

,feature_set,n_features,train_groupkfold3_mae,test_mae
0,baseline_20_features,20,76672.766966,102875.753900
1,sem_comps_knn_price_19_features,19,103111.121147,98687.565617


In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

variants = ["baseline_20_features", "sem_comps_knn_price_19_features"]
labels = ["baseline (20 features)", "sem comps_knn_price (19)"]
colors = ["#4C72B0", "#C44E52"]

train_maes = [report["train_groupkfold3"]["baseline_mae_mean"], report["train_groupkfold3"]["candidato_mae_mean"]]
test_maes = [report["test_holdout"]["baseline_mae"], report["test_holdout"]["candidato_mae"]]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes[0].bar(labels, train_maes, color=colors)
axes[0].set_title("MAE TRAIN GroupKFold(3)")
axes[0].set_ylabel("MAE ($)")

axes[1].bar(labels, test_maes, color=colors)
axes[1].set_title("MAE TEST (holdout)")

by_band = report["test_holdout"]["by_price_band"]
bands = list(by_band.keys())
x = range(len(bands))
axes[2].bar([i - 0.2 for i in x], [by_band[b]["mae_baseline"] for b in bands], width=0.4, label="baseline", color=colors[0])
axes[2].bar([i + 0.2 for i in x], [by_band[b]["mae_candidato"] for b in bands], width=0.4, label="sem comps_knn_price", color=colors[1])
axes[2].set_xticks(list(x)); axes[2].set_xticklabels(bands)
axes[2].legend()
axes[2].set_title("MAE TEST por price_band")

plt.tight_layout()
plt.savefig("reports/figures/12_drop_comps_knn_price_iter3.png", dpi=110)
plt.show()

print("decision:", report["decision"])

decision: REJECT_KEEP_COMPS_KNN_PRICE


C:\Users\br3nd\AppData\Local\Temp\ipykernel_34664\2863640244.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Análise

| Conjunto de features | MAE TRAIN GroupKFold(3) | MAE TEST (holdout) |
|---|---|---|
| Baseline (20 features, com `comps_knn_price`) | **\$76.906** | **\$104.221** |
| Sem `comps_knn_price` (19 features) | \$103.577 (+34,68%, piora) | \$99.015 (-4,99%) |

Por `price_band` em TEST (P3 — nunca decide só pelo número global):

| Banda | MAE baseline | MAE sem `comps_knn_price` | Variação |
|---|---|---|---|
| Luxury | \$207.942 | \$187.967 | -9,61% |
| Premium | \$121.504 | \$82.701 | -31,94% |
| Standard | \$76.122 | \$79.943 | +5,02% |
| Entry | \$56.522 | \$75.778 | **+34,07%** |

Remover `comps_knn_price` até melhora o MAE global de TEST (-4,99%) e melhora bastante `Luxury` e
`Premium` — mas **piora fortemente o `TRAIN GroupKFold(3)`** (+34,68%, viola a primeira condição do
critério, que exige não piorar) **e piora a banda `Entry` em +34,07%** (viola o limite de 10% por
banda, P3). O MAE de TEST também fica a -4,99%, um "near miss" que não atinge o corte decisivo de -5%
(mesmo padrão de near miss já visto na fase 12 iteração 1).

![drop comps_knn_price](../reports/figures/12_drop_comps_knn_price_iter3.png)

## Decisão

**`REJECT_KEEP_COMPS_KNN_PRICE`** — nenhuma das três condições do critério pré-registrado foi
totalmente satisfeita, e as duas primeiras já bastam para rejeitar (adoção exige TODAS): TRAIN piorou
(+34,68%, quando o critério exige não piorar), TEST melhorou mas não atingiu a queda decisiva de -5%
(-4,99%, near miss), e a banda `Entry` piorou +34,07% (> 10% tolerado). O ganho aparente na métrica
global de TEST (-4,99%) e nas bandas de luxo esconderia uma piora severa e concentrada em `Entry` —
exatamente o cenário que a disciplina de quebra por `price_band` (P3) existe para capturar antes de uma
decisão de promoção. `comps_knn_price` permanece no conjunto oficial de 20 features; nenhuma reexecução
de fase 10/13/14 é necessária (a hipótese não foi adotada).

## Artefatos

`scripts/run_hypothesis_drop_comps_knn_price.py`, `reports/hypothesis_drop_comps_knn_price.json`,
`reports/figures/12_drop_comps_knn_price_iter3.png`